# Tutorial for SHACL advanced Features used for the new Reac4Cat

the idea is to use SHACL as a body for a SPARQL Construct query, in which new relations between an Experiment and idealized reactions can be made to clearly state which conceptual reactions occurred in a Reaction. 

But to do so we need to consider and face some challenges.

First, SHACL as a base feature is not intended to modify its Data graph. SHACL-AF is allowed to do so, but still not necessarily intended to do so.
This means while you are allowed to modify your temporary graph with SHACL-AF, this does not necessarily mean that you want to permanently modify your graph.

As such, most tools typically don't give you a method of retrieving the modified graph. 

This is where [pySHACL](https://pyshacl.readthedocs.io/en/latest/) and this [issue](https://github.com/RDFLib/pySHACL/issues/189) come in handy.

In [ ]:
a function 

In [1]:
from pyshacl import Validator
from rdflib import Graph
from rdflib.compare import to_isomorphic, graph_diff


def gen_dif_and_ext_graph(data_g, shape_g):
    val_0 = Validator(data_g, options={"advanced": True, "inference": "rdfs"})
    conforms_data, report_g_data, report_text_data = val_0.run()
    inferred_base_graph_1 = val_0.target_graph
    val_1 = Validator(data_g, shacl_graph=shape_g, options={"advanced": True, "inference": "rdfs"})
    conforms_expanded, report_g_expanded, report_text_expanded = val_1.run()
    expanded_g = val_1.target_graph
    is0_1, is0_2 = to_isomorphic(inferred_base_graph_1), to_isomorphic(expanded_g)
    both, diff_g1, diff_g2 = graph_diff(is0_1, is0_2)
    conforms_sum = f'baseshape: \n{conforms_data}\nexpanded graph: \n{conforms_expanded}'
    dict_of_report_g = {'baseshape_report_graph': report_g_data, 'expanded_report_graph': report_g_expanded}
    report_text_sum = f'baseshape: \n{report_text_data}\nexpanded graph: \n{report_text_expanded}'
    return expanded_g, diff_g2, conforms_sum, dict_of_report_g, report_text_sum


# path_to_data = "D:/Users/smhhborg(D_Laufwerk)/Documents/GitHub/Test_for_advanced_SHACL_features/shacl_sparql_test_data.ttl"
# path_to_shapes = "D:/Users/smhhborg(D_Laufwerk)/Documents/GitHub/Test_for_advanced_SHACL_features/shacl_sparql_test_shape.ttl"

# data_graph = Graph()
# data_graph.parse(path_to_data, format="ttl")

# shape_graph = Graph()
# shape_graph.parse(path_to_shapes, format="ttl")

# expanded_graph, diff_graph, conforms, report_graph_dict, report_text = gen_dif_and_ext_graph(data_graph, shape_graph)
# expanded_graph.serialize(destination='./expanded_graph.ttl', format='turtle')
# diff_graph.serialize(destination='./diff_graph.ttl', format='turtle')

<Graph identifier=Nd08becebba584bfba269b2d21e3403b5 (<class 'rdflib.graph.Graph'>)>

the data should look like this:

In [2]:
data_set = 
"""
@prefix ro: <http://www.example.org/reaction-ontology#> .
@prefix ex: <http://www.example.org/chemicals#> .

# Idealized Reaction 1: Ethanol + Carbon dioxide -> Ethyl acetate
ex:IdealReaction1 a ro:Reaction ;
                  ro:type "idealized" ;
				  ex:hasInitialMixture ex:InitialMixture_i1;
				  ex:hasProductMixture ex:ProductMixture_i1 .
				  
ex:InitialMixture_i1 ro:hasReactant ex:Ethanol , ex:CarbonDioxide .
ex:ProductMixture_i1 ro:hasProduct ex:EthylAcetate .

# Idealized Reaction 2: Acetic acid -> Diethyl ether + Formate
ex:IdealReaction2 a ro:Reaction ;
                  ro:type "idealized" ;
				  ex:hasInitialMixture ex:InitialMixture_i2;
				  ex:hasProductMixture ex:ProductMixture_i2 .
				  
ex:InitialMixture_i2 ro:hasReactant ex:AceticAcid .
ex:ProductMixture_i2 ro:hasProduct ex:DiethylEther , ex:Formate .

# Idealized Reaction 3: Ethanol -> Ethylene glycol
ex:IdealReaction3 a ro:Reaction ;
                  ro:type "idealized" ;
				  ex:hasInitialMixture ex:InitialMixture_i3;
				  ex:hasProductMixture ex:ProductMixture_i3 .
				  
ex:InitialMixture_i3 ro:hasReactant ex:Ethanol .
ex:ProductMixture_i3 ro:hasProduct ex:EthyleneGlycol .

# Experimental Reaction 1: Matches Idealized Reaction 1
ex:ExperimentalReaction1 a ro:Reaction ;
                         ro:type "experimental" ;
						 ex:hasInitialMixture ex:InitialMixture_e1;
						 ex:hasProductMixture ex:ProductMixture_e1 .
						 
ex:InitialMixture_e1 ro:hasReactant ex:Ethanol , ex:CarbonDioxide .
ex:ProductMixture_e1 ro:hasProduct ex:EthylAcetate .

# Experimental Reaction 2: Matches Idealized Reaction 2
ex:ExperimentalReaction2 a ro:Reaction ;
                         ro:type "experimental" ;
						 ex:hasInitialMixture ex:InitialMixture_e2;
						 ex:hasProductMixture ex:ProductMixture_e3 .
						 
ex:InitialMixture_e2 ro:hasReactant ex:AceticAcid, ex:Methanol .
ex:ProductMixture_e3 ro:hasProduct ex:DiethylEther , ex:Formate .

# Experimental Reaction 3: Matches Idealized Reaction 1 with additional reactant
ex:ExperimentalReaction3 a ro:Reaction ;
                         ro:type "experimental" ;						
						 ex:hasInitialMixture ex:InitialMixture_e3;
						 ex:hasProductMixture ex:ProductMixture_e1 .

ex:InitialMixture_e3 ro:hasReactant ex:Ethanol , ex:CarbonDioxide , ex:Methanol .

# Experimental Reaction 4: Matches Idealized Reaction 1 with incorrect product
ex:ExperimentalReaction4 a ro:Reaction ;
                         ro:type "experimental" ;						 
						 ex:hasInitialMixture ex:InitialMixture_e3;
						 ex:hasProductMixture ex:ProductMixture_e4 .
						 
ex:ProductMixture_e4 ro:hasProduct ex:DiethylEther .

# Experimental Reaction 5: Matches Idealized Reaction 3
ex:ExperimentalReaction5 a ro:Reaction ;
                         ro:type "experimental" ;
						 ex:hasInitialMixture ex:InitialMixture_e4;
						 ex:hasProductMixture ex:ProductMixture_e5 .

ex:InitialMixture_e4 ro:hasReactant ex:Ethanol .
ex:ProductMixture_e5 ro:hasProduct ex:EthyleneGlycol .

# Experimental Reaction 6: Non-matching reaction
ex:ExperimentalReaction6 a ro:Reaction ;
                         ro:type "experimental" ;
						 ex:hasInitialMixture ex:InitialMixture_e5;
						 ex:hasProductMixture ex:ProductMixture_e2 .
						 
ex:InitialMixture_e5 ro:hasReactant ex:Methanol , ex:CarbonDioxide .
"""


In [8]:
data_graph = Graph()
data_graph.parse(data=data_set, format="turtle")

<Graph identifier=N6359c25825b8407f93db68694056aef2 (<class 'rdflib.graph.Graph'>)>